# Complete Transfer Learning: Motor Execution → Motor Imagery

Comprehensive transfer learning study across all new model architectures
and channel configurations.

**Models:** EEGSimpleConv, EEGInceptionMI, EEGSym, MSVTNet, EEGNeX, CTNet

**Conditions:**
1. Imagery only (baseline)
2. Movement → Imagery (zero-shot transfer)
3. Movement pretrain + imagery fine-tune

**Channel ablation:** all 64, sensorimotor-17, sensorimotor-9

**Evaluation:** 5 random group-based splits, balanced accuracy

In [1]:
import random
import time
import warnings
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display
from braindecode.models import (
    CTNet,
    EEGInceptionMI,
    EEGNeX,
    EEGSimpleConv,
    EEGSym,
    MSVTNet,
)
from mne import Epochs, pick_types
from mne.channels import make_standard_montage
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("ERROR")
plt.style.use("seaborn-v0_8-whitegrid")

/home/aniruddham/code/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
SEED = 7
DATA_ROOT = Path("/NAS/aniruddham/mne/data")
MOVEMENT_RUNS = [3, 7, 11]  # actual left/right fist clenching
IMAGERY_RUNS = [4, 8, 12]  # imagined left/right hand movement
RESAMPLE_SFREQ = 160.0
FILTER_BAND = (8.0, 30.0)
EPOCH_WINDOW = (0.0, 4.0)
CROP_WINDOW = (0.5, 2.5)

CHANNEL_OPTIONS = {
    "all_64": None,
    "sensorimotor_17": [
        "FC5",
        "FC3",
        "FC1",
        "FCz",
        "FC2",
        "FC4",
        "FC6",
        "C5",
        "C3",
        "C1",
        "Cz",
        "C2",
        "C4",
        "C6",
        "CP3",
        "CPz",
        "CP4",
    ],
    "sensorimotor_9": [
        "FC3",
        "FCz",
        "FC4",
        "C3",
        "Cz",
        "C4",
        "CP3",
        "CPz",
        "CP4",
    ],
}

MODEL_NAMES = [
    "EEGSimpleConv",
    "EEGInceptionMI",
    "EEGSym",
    "MSVTNet",
    "EEGNeX",
    "CTNet",
]

N_SPLITS = 5
MAX_SUBJECTS = None
BATCH_SIZE = 64
PRETRAIN_EPOCHS = 12
FINETUNE_EPOCHS = 8
FINETUNE_LR = 5e-4
BASELINE_EPOCHS = 12
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
TEST_SIZE = 0.2
VAL_SIZE = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MONTAGE = make_standard_montage("standard_1005")

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print(f"Using device: {DEVICE}")

Using device: cuda


## Data loading (movement + imagery)

In [3]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def infer_available_subjects(data_root: Path = DATA_ROOT) -> list[int]:
    return sorted(
        int(path.name[1:])
        for path in data_root.glob("S[0-9][0-9][0-9]")
        if path.is_dir()
    )


def local_run_paths(
    subject: int, runs: list[int], data_root: Path = DATA_ROOT
) -> list[Path]:
    subject_name = f"S{subject:03d}"
    return [data_root / subject_name / f"{subject_name}R{run:02d}.edf" for run in runs]


def load_subject_raw(subject: int, runs: list[int]) -> mne.io.BaseRaw:
    paths = local_run_paths(subject, runs)
    raw = concatenate_raws(
        [read_raw_edf(path, preload=True, verbose="ERROR") for path in paths]
    )
    eegbci.standardize(raw)
    raw.pick("eeg")
    raw.set_montage(MONTAGE, match_case=False, on_missing="warn")
    raw.annotations.rename({"T1": "left_hand", "T2": "right_hand"})
    raw.set_eeg_reference("average", projection=False, verbose="ERROR")
    raw.resample(RESAMPLE_SFREQ, verbose="ERROR")
    raw.filter(
        *FILTER_BAND, fir_design="firwin", skip_by_annotation="edge", verbose="ERROR"
    )
    return raw


def epoch_subject(raw: mne.io.BaseRaw) -> tuple[np.ndarray, np.ndarray, Epochs]:
    events, event_map = mne.events_from_annotations(raw, verbose="ERROR")
    epochs = Epochs(
        raw,
        events,
        event_id={
            "left_hand": event_map["left_hand"],
            "right_hand": event_map["right_hand"],
        },
        tmin=EPOCH_WINDOW[0],
        tmax=EPOCH_WINDOW[1],
        proj=False,
        picks=pick_types(
            raw.info, meg=False, eeg=True, stim=False, eog=False, exclude="bads"
        ),
        baseline=None,
        preload=True,
        reject_by_annotation=True,
        verbose="ERROR",
    ).crop(*CROP_WINDOW)
    if len(epochs.times) % 2 != 0:
        epochs.crop(tmin=epochs.tmin, tmax=epochs.times[-2])
    X = epochs.get_data(copy=False).astype(np.float32)
    y = (epochs.events[:, -1] == event_map["right_hand"]).astype(np.int64)
    return X, y, epochs


def select_channels(X, channel_names, chs_info, wanted):
    if wanted is None:
        return X, channel_names, chs_info
    name_to_idx = {name: idx for idx, name in enumerate(channel_names)}
    picks = [name_to_idx[ch] for ch in wanted if ch in name_to_idx]
    return (
        X[:, picks, :],
        [channel_names[idx] for idx in picks],
        [deepcopy(chs_info[idx]) for idx in picks],
    )


def build_dataset(runs: list[int], label: str) -> dict:
    subjects = infer_available_subjects()
    if MAX_SUBJECTS is not None:
        subjects = subjects[:MAX_SUBJECTS]

    X_parts, y_parts, groups = [], [], []
    channel_names = None
    chs_info = None

    for subject in tqdm(subjects, desc=f"Loading {label}"):
        raw = load_subject_raw(subject, runs)
        X, y, epochs = epoch_subject(raw)
        mean = X.mean(axis=2, keepdims=True)
        std = X.std(axis=2, keepdims=True) + 1e-6
        X = (X - mean) / std

        if channel_names is None:
            channel_names = epochs.ch_names
            chs_info = deepcopy(epochs.info["chs"])

        X_parts.append(X)
        y_parts.append(y)
        groups.append(np.full(len(y), subject))

    return {
        "X": np.concatenate(X_parts, axis=0),
        "y": np.concatenate(y_parts, axis=0),
        "groups": np.concatenate(groups, axis=0),
        "channel_names": list(channel_names),
        "chs_info": chs_info,
        "sfreq": RESAMPLE_SFREQ,
    }


set_seed()
imagery_ds = build_dataset(IMAGERY_RUNS, "imagery")
movement_ds = build_dataset(MOVEMENT_RUNS, "movement")

print(
    f"Imagery:  {len(imagery_ds['y'])} epochs, {len(np.unique(imagery_ds['groups']))} subjects"
)
print(
    f"Movement: {len(movement_ds['y'])} epochs, {len(np.unique(movement_ds['groups']))} subjects"
)

Loading movement: 100%|██████████| 109/109 [01:17<00:00,  1.40it/s]


Imagery:  4898 epochs, 109 subjects
Movement: 4897 epochs, 109 subjects


## Training helpers

In [4]:
def make_group_split(groups: np.ndarray, seed: int) -> dict:
    index = np.arange(len(groups))
    outer = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
    train_val_idx, test_idx = next(outer.split(index, groups=groups))
    inner = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=seed + 100)
    inner_train_rel, inner_val_rel = next(
        inner.split(np.arange(len(train_val_idx)), groups=groups[train_val_idx])
    )
    return {
        "train": train_val_idx[inner_train_rel],
        "val": train_val_idx[inner_val_rel],
        "test": test_idx,
    }


def make_loader(X: np.ndarray, y: np.ndarray, shuffle: bool) -> DataLoader:
    return DataLoader(
        TensorDataset(torch.from_numpy(X), torch.from_numpy(y)),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        drop_last=False,
    )


def unpack_logits(logits):
    if isinstance(logits, (tuple, list)):
        logits = logits[0]
    while logits.ndim > 2 and logits.shape[-1] == 1:
        logits = logits.squeeze(-1)
    if logits.ndim > 2:
        logits = logits.mean(dim=tuple(range(2, logits.ndim)))
    return logits


@torch.no_grad()
def predict_torch(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    preds = []
    for batch_X, _ in loader:
        batch_X = batch_X.to(DEVICE)
        logits = unpack_logits(model(batch_X))
        preds.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(preds)


def build_model(
    model_name: str, n_chans: int, n_times: int, sfreq: float, chs_info: list[dict]
) -> nn.Module:
    common = dict(
        n_chans=n_chans, n_times=n_times, n_outputs=2, sfreq=sfreq, chs_info=chs_info
    )
    if model_name == "EEGSimpleConv":
        return EEGSimpleConv(**common).to(DEVICE)
    if model_name == "EEGInceptionMI":
        return EEGInceptionMI(**common).to(DEVICE)
    if model_name == "EEGSym":
        return EEGSym(**common, scales_time=(525, 275, 125)).to(DEVICE)
    if model_name == "MSVTNet":
        return MSVTNet(**common).to(DEVICE)
    if model_name == "EEGNeX":
        return EEGNeX(**common).to(DEVICE)
    if model_name == "CTNet":
        return CTNet(**common).to(DEVICE)
    raise KeyError(model_name)


def train_phase(model, train_loader, val_loader, y_val, max_epochs, lr=LEARNING_RATE):
    """Train with early stopping, return best model state."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    best_state = None
    best_val_bal = -np.inf
    best_epoch = 0
    patience_counter = 0
    start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = unpack_logits(model(batch_X))
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

        val_pred = predict_torch(model, val_loader)
        val_bal = balanced_accuracy_score(y_val, val_pred)
        if val_bal > best_val_bal + 1e-4:
            best_val_bal = val_bal
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    elapsed = time.perf_counter() - start
    return model, best_epoch, elapsed

## Run transfer experiments (6 models x 3 channels x 3 conditions x 5 splits)

In [5]:
all_rows = []
total_runs = len(CHANNEL_OPTIONS) * N_SPLITS * len(MODEL_NAMES) * 3  # 3 conditions
pbar = tqdm(total=total_runs, desc="Transfer experiments")

for subset_name, subset_channels in CHANNEL_OPTIONS.items():
    # Select channels for both datasets
    Xi, chi_names, chi_info = select_channels(
        imagery_ds["X"],
        imagery_ds["channel_names"],
        imagery_ds["chs_info"],
        subset_channels,
    )
    Xm, _, _ = select_channels(
        movement_ds["X"],
        movement_ds["channel_names"],
        movement_ds["chs_info"],
        subset_channels,
    )
    n_chans, n_times, sfreq = Xi.shape[1], Xi.shape[2], imagery_ds["sfreq"]

    for split_idx in range(N_SPLITS):
        seed = SEED + split_idx

        # Same subject split for both datasets
        i_split = make_group_split(imagery_ds["groups"], seed)
        m_split = make_group_split(movement_ds["groups"], seed)

        Xi_train, yi_train = Xi[i_split["train"]], imagery_ds["y"][i_split["train"]]
        Xi_val, yi_val = Xi[i_split["val"]], imagery_ds["y"][i_split["val"]]
        Xi_test, yi_test = Xi[i_split["test"]], imagery_ds["y"][i_split["test"]]

        Xm_train, ym_train = Xm[m_split["train"]], movement_ds["y"][m_split["train"]]
        Xm_val, ym_val = Xm[m_split["val"]], movement_ds["y"][m_split["val"]]

        i_train_loader = make_loader(Xi_train, yi_train, shuffle=True)
        i_val_loader = make_loader(Xi_val, yi_val, shuffle=False)
        i_test_loader = make_loader(Xi_test, yi_test, shuffle=False)
        m_train_loader = make_loader(Xm_train, ym_train, shuffle=True)
        m_val_loader = make_loader(Xm_val, ym_val, shuffle=False)

        for model_name in MODEL_NAMES:
            desc = f"{model_name} | {subset_name} | split {split_idx}"

            # --- Condition 1: Imagery only ---
            pbar.set_postfix_str(f"{desc} | imagery_only")
            set_seed(seed)
            model = build_model(model_name, n_chans, n_times, sfreq, chi_info)
            model, best_ep, elapsed = train_phase(
                model, i_train_loader, i_val_loader, yi_val, BASELINE_EPOCHS
            )
            test_pred = predict_torch(model, i_test_loader)
            all_rows.append(
                {
                    "model": model_name,
                    "channel_subset": subset_name,
                    "n_channels": n_chans,
                    "split": split_idx,
                    "condition": "imagery_only",
                    "test_balanced_accuracy": balanced_accuracy_score(
                        yi_test, test_pred
                    ),
                    "best_epoch": best_ep,
                    "train_time_s": elapsed,
                }
            )
            pbar.update(1)

            # --- Condition 2: Movement → Imagery (zero-shot) ---
            pbar.set_postfix_str(f"{desc} | zero_shot")
            set_seed(seed)
            model = build_model(model_name, n_chans, n_times, sfreq, chi_info)
            model, best_ep, elapsed = train_phase(
                model, m_train_loader, m_val_loader, ym_val, PRETRAIN_EPOCHS
            )
            test_pred = predict_torch(model, i_test_loader)
            all_rows.append(
                {
                    "model": model_name,
                    "channel_subset": subset_name,
                    "n_channels": n_chans,
                    "split": split_idx,
                    "condition": "zero_shot",
                    "test_balanced_accuracy": balanced_accuracy_score(
                        yi_test, test_pred
                    ),
                    "best_epoch": best_ep,
                    "train_time_s": elapsed,
                }
            )
            pbar.update(1)

            # --- Condition 3: Movement pretrain + imagery fine-tune ---
            pbar.set_postfix_str(f"{desc} | pretrain+finetune")
            set_seed(seed)
            model = build_model(model_name, n_chans, n_times, sfreq, chi_info)
            model, _, pre_time = train_phase(
                model, m_train_loader, m_val_loader, ym_val, PRETRAIN_EPOCHS
            )
            model, ft_ep, ft_time = train_phase(
                model,
                i_train_loader,
                i_val_loader,
                yi_val,
                FINETUNE_EPOCHS,
                lr=FINETUNE_LR,
            )
            test_pred = predict_torch(model, i_test_loader)
            all_rows.append(
                {
                    "model": model_name,
                    "channel_subset": subset_name,
                    "n_channels": n_chans,
                    "split": split_idx,
                    "condition": "pretrain_finetune",
                    "test_balanced_accuracy": balanced_accuracy_score(
                        yi_test, test_pred
                    ),
                    "best_epoch": ft_ep,
                    "train_time_s": pre_time + ft_time,
                }
            )
            pbar.update(1)

pbar.close()
results_df = pd.DataFrame(all_rows)
print(f"Collected {len(results_df)} results.")
display(results_df.head(12))

Transfer experiments:   1%|          | 3/270 [00:28<34:41,  7.80s/it, EEGInceptionMI | all_64 | split 0 | imagery_only]    /home/aniruddham/code/project/.venv/lib/python3.14/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(
Transfer experiments:   3%|▎         | 9/270 [08:44<7:22:05, 101.63s/it, MSVTNet | all_64 | split 0 | imagery_only]           /home/aniruddham/code/project/.venv/lib/python3.14/site-packages/braindecode/models/msvtnet.py:340: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(
Transfer experiments:   4%|▎         | 10/270 [09:07<5:35:13, 77.36s/it, MSVTNet | all_64 | split 0 | zero_shot]   /home/aniruddham/code/project/.venv/lib/python3.14/sit

KeyboardInterrupt: 

## Summary table

In [ ]:
summary = results_df.groupby(
    ["model", "channel_subset", "n_channels", "condition"], as_index=False
).agg(
    mean_bal_acc=("test_balanced_accuracy", "mean"),
    std_bal_acc=("test_balanced_accuracy", "std"),
    mean_train_time_s=("train_time_s", "mean"),
)

# Pivot: one row per model+channel, columns for each condition
pivot = summary.pivot_table(
    index=["model", "channel_subset", "n_channels"],
    columns="condition",
    values="mean_bal_acc",
).reset_index()
pivot.columns.name = None
pivot["zero_shot_delta"] = pivot["zero_shot"] - pivot["imagery_only"]
pivot["finetune_delta"] = pivot["pretrain_finetune"] - pivot["imagery_only"]
pivot = pivot.sort_values("imagery_only", ascending=False).reset_index(drop=True)
display(pivot)

## Transfer comparison per model (bar charts by channel config)

In [ ]:
channel_order = ["all_64", "sensorimotor_17", "sensorimotor_9"]
conditions = ["imagery_only", "zero_shot", "pretrain_finetune"]
cond_labels = ["Imagery only", "Zero-shot transfer", "Pretrain + finetune"]
cond_colors = ["#4477AA", "#CC6677", "#228833"]

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
width = 0.25
x = np.arange(len(conditions))

for idx, model_name in enumerate(MODEL_NAMES):
    ax = axes[idx // 3, idx % 3]
    for j, ch_sub in enumerate(channel_order):
        vals = []
        stds = []
        for cond in conditions:
            row = summary[
                (summary["model"] == model_name)
                & (summary["channel_subset"] == ch_sub)
                & (summary["condition"] == cond)
            ]
            vals.append(row["mean_bal_acc"].values[0] if len(row) > 0 else 0)
            stds.append(row["std_bal_acc"].values[0] if len(row) > 0 else 0)
        offset = (j - 1) * width
        bars = ax.bar(
            x + offset,
            vals,
            width,
            yerr=stds,
            capsize=3,
            label=ch_sub,
            edgecolor="black",
            linewidth=0.5,
            color=["#4477AA", "#228833", "#CCBB44"][j],
            alpha=0.85,
        )
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.035,
                f"{v:.3f}",
                ha="center",
                va="bottom",
                fontsize=7,
                fontweight="bold",
            )

    ax.axhline(0.5, color="grey", linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(cond_labels, fontsize=9)
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_ylabel("Balanced accuracy")
    ax.set_ylim(0.40, 0.75)
    ax.legend(fontsize=8, loc="upper right")

plt.suptitle(
    "Transfer learning: 3 conditions x 3 channel configs per model (5 splits)",
    fontsize=15,
    fontweight="bold",
    y=0.99,
)
plt.tight_layout()
plt.show()

## Transfer gain/loss heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax_idx, (delta_col, title) in enumerate(
    [
        ("zero_shot_delta", "Zero-shot transfer − Imagery only"),
        ("finetune_delta", "Pretrain+finetune − Imagery only"),
    ]
):
    # Build matrix: rows=models, cols=channel configs
    matrix = np.zeros((len(MODEL_NAMES), len(channel_order)))
    for i, m in enumerate(MODEL_NAMES):
        for j, ch in enumerate(channel_order):
            row = pivot[(pivot["model"] == m) & (pivot["channel_subset"] == ch)]
            if len(row) > 0:
                matrix[i, j] = row[delta_col].values[0]

    vlim = max(abs(matrix.min()), abs(matrix.max()))
    vlim = max(vlim, 0.02)
    im = axes[ax_idx].imshow(
        matrix, cmap="RdBu_r", aspect="auto", vmin=-vlim, vmax=vlim
    )
    axes[ax_idx].set_xticks(range(len(channel_order)))
    axes[ax_idx].set_xticklabels(["All 64", "SM-17", "SM-9"], fontsize=12)
    axes[ax_idx].set_yticks(range(len(MODEL_NAMES)))
    axes[ax_idx].set_yticklabels(MODEL_NAMES, fontsize=12)
    for i in range(len(MODEL_NAMES)):
        for j in range(len(channel_order)):
            val = matrix[i, j]
            sign = "+" if val > 0 else ""
            color = "white" if abs(val) > vlim * 0.55 else "black"
            axes[ax_idx].text(
                j,
                i,
                f"{sign}{val:.3f}",
                ha="center",
                va="center",
                fontsize=12,
                fontweight="bold",
                color=color,
            )
    plt.colorbar(im, ax=axes[ax_idx], fraction=0.046, pad=0.04)
    axes[ax_idx].set_title(title, fontsize=13, fontweight="bold")

plt.suptitle(
    "Transfer learning gain/loss vs imagery-only baseline",
    fontsize=15,
    fontweight="bold",
    y=0.99,
)
plt.tight_layout()
plt.show()

## Per-channel bar chart (all models side by side)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7), sharey=True)
colors_list = ["#228833", "#CCBB44", "#AA3377", "#EE7733", "#0077BB", "#332288"]
width = 0.12

for col_idx, ch_sub in enumerate(channel_order):
    ax = axes[col_idx]
    ch_label = {
        "all_64": "All 64",
        "sensorimotor_17": "SM-17",
        "sensorimotor_9": "SM-9",
    }[ch_sub]
    x = np.arange(len(conditions))

    for i, model_name in enumerate(MODEL_NAMES):
        vals = []
        for cond in conditions:
            row = summary[
                (summary["model"] == model_name)
                & (summary["channel_subset"] == ch_sub)
                & (summary["condition"] == cond)
            ]
            vals.append(row["mean_bal_acc"].values[0] if len(row) > 0 else 0)
        offset = (i - len(MODEL_NAMES) / 2 + 0.5) * width
        ax.bar(
            x + offset,
            vals,
            width,
            label=model_name,
            color=colors_list[i],
            edgecolor="black",
            linewidth=0.5,
        )

    ax.axhline(0.5, color="grey", linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(cond_labels, fontsize=10)
    ax.set_title(f"{ch_label}", fontsize=14, fontweight="bold")
    ax.set_ylabel("Balanced accuracy" if col_idx == 0 else "")
    ax.set_ylim(0.40, 0.75)
    if col_idx == 2:
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)

plt.suptitle(
    "All models x conditions, grouped by channel config (5 splits)",
    fontsize=15,
    fontweight="bold",
    y=0.99,
)
plt.tight_layout()
plt.show()

## Box plots: per-split distributions by condition

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))

for idx, model_name in enumerate(MODEL_NAMES):
    ax = axes[idx // 3, idx % 3]
    model_df = results_df[results_df["model"] == model_name]

    positions = []
    data = []
    tick_labels = []
    colors_box = []
    pos = 0
    cond_palette = {
        "imagery_only": "#4477AA",
        "zero_shot": "#CC6677",
        "pretrain_finetune": "#228833",
    }

    for ch_sub in channel_order:
        for cond in conditions:
            subset = model_df[
                (model_df["channel_subset"] == ch_sub) & (model_df["condition"] == cond)
            ]
            data.append(subset["test_balanced_accuracy"].values)
            positions.append(pos)
            colors_box.append(cond_palette[cond])
            pos += 1
        pos += 0.5  # gap between channel groups

    bp = ax.boxplot(
        data,
        positions=positions,
        patch_artist=True,
        widths=0.7,
        showmeans=True,
        meanprops=dict(marker="D", markerfacecolor="red", markersize=5),
    )
    for patch, color in zip(bp["boxes"], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.axhline(0.5, color="grey", linestyle="--", linewidth=1)
    # Group labels
    group_centers = [1, 4.5, 8]
    ax.set_xticks(group_centers)
    ax.set_xticklabels(["All 64", "SM-17", "SM-9"], fontsize=11)
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_ylabel("Balanced accuracy")
    ax.set_ylim(0.35, 0.80)

# Shared legend
from matplotlib.patches import Patch

legend_patches = [
    Patch(facecolor=cond_palette[c], label=l, alpha=0.7)
    for c, l in zip(conditions, cond_labels)
]
fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=3,
    fontsize=11,
    bbox_to_anchor=(0.5, -0.02),
)
plt.suptitle(
    "Per-split accuracy distributions by condition", fontsize=15, fontweight="bold"
)
plt.tight_layout()
plt.show()

## Best configuration and conclusions

In [ ]:
# How often does each condition win?
wins = {"imagery_only": 0, "zero_shot": 0, "pretrain_finetune": 0}
for _, group in pivot.iterrows():
    best_cond = max(conditions, key=lambda c: group.get(c, 0))
    wins[best_cond] += 1

print("=" * 60)
print("TRANSFER LEARNING SUMMARY")
print("=" * 60)
print(f"\nCondition win counts (across {len(pivot)} model x channel configs):")
for cond, count in wins.items():
    print(f"  {cond:25s}: {count}/{len(pivot)}")

print(f"\nMean zero-shot delta:       {pivot['zero_shot_delta'].mean():+.4f}")
print(f"Mean finetune delta:        {pivot['finetune_delta'].mean():+.4f}")

# Best overall config
best_row = pivot.loc[pivot["imagery_only"].idxmax()]
print(f"\nBest imagery-only config:")
print(f"  Model:          {best_row['model']}")
print(f"  Channels:       {best_row['channel_subset']}")
print(f"  Imagery-only:   {best_row['imagery_only']:.4f}")
print(
    f"  Zero-shot:      {best_row['zero_shot']:.4f} ({best_row['zero_shot_delta']:+.4f})"
)
print(
    f"  Finetune:       {best_row['pretrain_finetune']:.4f} ({best_row['finetune_delta']:+.4f})"
)
print("=" * 60)

# Top 5 configs by imagery-only
print("\nTop 5 configurations (imagery-only):")
display(
    pivot.head(5)[
        [
            "model",
            "channel_subset",
            "imagery_only",
            "zero_shot",
            "pretrain_finetune",
            "zero_shot_delta",
            "finetune_delta",
        ]
    ]
)